# SHAP + reconciliation diagnostic — fully self-sufficient, no prior session needed

**Why this is much smaller than the full Stage-4-v2 pipeline**: SHAP and the
reconciliation diagnostic only need the stylometric features for `train` and `testA`
— they do NOT need BERT/DistilBERT embeddings, the attention-gated model, or any
training loop at all. This notebook builds exactly what those two things need, nothing
more, from scratch.

**Run this top to bottom in a fresh Kaggle session.** GPU is optional here (helps GPT-2
perplexity computation go faster, but nothing in this notebook requires it strictly).

**What it does NOT recreate**: any of the Hybrid/GLTR/fine-tuned-transformer training
or comparison work from Stage-4-v2 — that's unaffected by anything here and doesn't
need to be rerun.

## 1. Setup + checkpoint infrastructure

In [ ]:
!pip install -q transformers scikit-learn pandas numpy nltk tqdm matplotlib shap

import os, re, json, random, warnings, glob, shutil, pickle
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
from nltk.corpus import stopwords

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '(CPU is fine for this notebook, just slower)')

CHECKPOINT_DIR = '/kaggle/working/checkpoints_shap'
ARTIFACTS_DIR = '/kaggle/working/artifacts_shap'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Restore from a previous attempt at THIS notebook, if any (harmless no-op on a first run)
restored = 0
for candidate in glob.glob('/kaggle/input/*/checkpoints_shap') + glob.glob('/kaggle/input/*/*/checkpoints_shap'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(CHECKPOINT_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(CHECKPOINT_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
print(f'Restored {restored} item(s) from a previous run of this notebook.')

def ckpt_path(name): return os.path.join(CHECKPOINT_DIR, name + '.pkl')
def ckpt_exists(name): return os.path.exists(ckpt_path(name))
def ckpt_save(name, obj):
    tmp = ckpt_path(name) + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(obj, f)
    os.replace(tmp, ckpt_path(name))
def ckpt_load(name):
    with open(ckpt_path(name), 'rb') as f: return pickle.load(f)

def find_artifact(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

STOPWORDS = set(stopwords.words('english'))
FUNCTION_WORDS = ['the', 'of', 'and', 'to', 'in', 'is', 'that', 'it']


## 2. Load M4 (only what's needed: train domains + peerread for testA)

Same verified loader as every other notebook in this project — bloomz field fix,
sorted file order for determinism.

In [ ]:
if not os.path.exists('/kaggle/working/M4'):
    !git clone -q https://github.com/mbzuai-nlp/M4.git /kaggle/working/M4
M4_ROOT = '/kaggle/working/M4/data'
MIN_WORDS = 30

def _wc(t): return len(t.split())
def clean(t): return re.sub(r'\s+', ' ', str(t)).strip()

def load_standard(path, domain, generator):
    human_rows, ai_rows = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: r = json.loads(line)
            except json.JSONDecodeError: continue
            ht, mt = clean(r.get('human_text', '')), clean(r.get('machine_text', ''))
            if _wc(ht) >= MIN_WORDS: human_rows.append({'text': ht, 'domain': domain, 'generator': 'human'})
            if _wc(mt) >= MIN_WORDS: ai_rows.append({'text': mt, 'domain': domain, 'generator': generator})
    return human_rows, ai_rows

def load_bloomz(path, domain):
    human_field = 'abstract' if domain == 'arxiv' else 'text'
    machine_field = 'machine_answer' if domain == 'reddit' else 'machine_abstract'
    human_rows, ai_rows = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: r = json.loads(line)
            except json.JSONDecodeError: continue
            ht, mt = clean(r.get(human_field, '')), clean(r.get(machine_field, ''))
            if _wc(ht) >= MIN_WORDS: human_rows.append({'text': ht, 'domain': domain, 'generator': 'human'})
            if _wc(mt) >= MIN_WORDS: ai_rows.append({'text': mt, 'domain': domain, 'generator': 'bloomz'})
    return human_rows, ai_rows

# Only the domains/generators actually needed for train + testA (peerread, core generators)
DOMAIN_FILES = {
    'arxiv':     {'chatGPT':'arxiv_chatGPT.jsonl','cohere':'arxiv_cohere.jsonl','davinci':'arxiv_davinci.jsonl',
                  'bloomz':'arxiv_bloomz.jsonl','flant5':'arxiv_flant5.jsonl'},
    'wikihow':   {'chatGPT':'wikihow_chatGPT.jsonl','cohere':'wikihow_cohere.jsonl','davinci':'wikihow_davinci.jsonl',
                  'bloomz':'wikihow_bloomz.jsonl'},
    'wikipedia': {'chatGPT':'wikipedia_chatgpt.jsonl','cohere':'wikipedia_cohere.jsonl','davinci':'wikipedia_davinci.jsonl',
                  'bloomz':'wikipedia_bloomz.jsonl'},
    'peerread':  {'chatGPT':'peerread_chatgpt.jsonl','cohere':'peerread_cohere.jsonl','davinci':'peerread_davinci.jsonl'},
}

if ckpt_exists('m4_raw_shap'):
    df_m4 = ckpt_load('m4_raw_shap')
    print('Loaded M4 (reduced) from checkpoint.')
else:
    all_human, all_ai = [], []
    for domain, gens in DOMAIN_FILES.items():
        for gen, fname in sorted(gens.items()):
            path = os.path.join(M4_ROOT, fname)
            if not os.path.exists(path): continue
            h, a = (load_bloomz(path, domain) if gen == 'bloomz' else load_standard(path, domain, gen))
            all_human.extend(h); all_ai.extend(a)
    human_df = pd.DataFrame(all_human).drop_duplicates(subset=['domain', 'text']).copy()
    ai_df = pd.DataFrame(all_ai)
    human_df['label'] = 0; ai_df['label'] = 1
    df_m4 = pd.concat([human_df, ai_df], ignore_index=True)
    df_m4 = df_m4.drop_duplicates(subset=['domain', 'generator', 'text']).reset_index(drop=True)
    ckpt_save('m4_raw_shap', df_m4)

print('Loaded rows:', len(df_m4))
print(pd.crosstab(df_m4.domain, df_m4.label))


## 3. Build `train` and `testA` — same split definition as Stage-4-v2

In [ ]:
M4_SEED = 42
TRAIN_DOMAINS = ['arxiv', 'wikihow', 'wikipedia']; TEST_DOMAIN = 'peerread'
TRAIN_GENS = ['chatGPT', 'cohere', 'davinci', 'bloomz', 'flant5']
N_TRAIN, N_TESTA = 4000, 500

def sample_n(pool, n, seed):
    if len(pool) < n: raise ValueError(f'Requested {n}, pool has {len(pool)}')
    return pool.sample(n=n, random_state=seed).reset_index(drop=True)

train_pool = df_m4[df_m4.domain.isin(TRAIN_DOMAINS)]
train_human = sample_n(train_pool[train_pool.label==0], N_TRAIN, M4_SEED)
train_ai = sample_n(train_pool[(train_pool.label==1)&(train_pool.generator.isin(TRAIN_GENS))], N_TRAIN, M4_SEED)
df_train = pd.concat([train_human, train_ai], ignore_index=True).sample(frac=1, random_state=M4_SEED).reset_index(drop=True)

test_pool = df_m4[df_m4.domain==TEST_DOMAIN]
testA_human = sample_n(test_pool[test_pool.label==0], N_TESTA, M4_SEED+2)
testA_ai = sample_n(test_pool[(test_pool.label==1)&(test_pool.generator.isin(TRAIN_GENS))], N_TESTA, M4_SEED+2)
df_testA = pd.concat([testA_human, testA_ai], ignore_index=True).sample(frac=1, random_state=M4_SEED+2).reset_index(drop=True)

overlap = set(df_train['text']) & set(df_testA['text'])
assert len(overlap) == 0, f'LEAKAGE: {len(overlap)} texts in both train and testA'
print('train:', df_train.shape, df_train.label.value_counts().to_dict())
print('testA:', df_testA.shape, df_testA.label.value_counts().to_dict())
print('No leakage between train and testA.')

y_train, y_testA = df_train.label.values, df_testA.label.values


## 4. Stylometric features — same self-consistent extractor as Stage-4-v2

Fit once on `df_train`, applied to both `train` and `testA`. This is the actual data
SHAP needs — everything above was just building up to this.

In [ ]:
def lexical_features(tokens):
    n = len(tokens)
    if n == 0: return {'hapax_ratio':0.,'yules_k':0.,'ttr':0.,'avg_word_len':0.}
    freqs = Counter(tokens); V = len(freqs)
    hapax = sum(1 for w,c in freqs.items() if c==1)
    freq_of_freq = Counter(freqs.values())
    sum_i2fi = sum((i**2)*fi for i,fi in freq_of_freq.items())
    return {'hapax_ratio':hapax/n,'yules_k':1e4*(sum_i2fi-n)/(n**2),'ttr':V/n,
            'avg_word_len':float(np.mean([len(w) for w in tokens]))}
def syntactic_features(sentences):
    lens = [len(word_tokenize(s)) for s in sentences] if sentences else [0]
    mean_len, var_len = float(np.mean(lens)), float(np.var(lens))
    burstiness = (var_len-mean_len)/(var_len+mean_len) if (var_len+mean_len)>0 else 0.
    full_text = ' '.join(sentences); n_chars = max(len(full_text),1)
    n_punct = sum(1 for c in full_text if c in '.,;:!?')
    return {'sent_len_variance':var_len,'burstiness':burstiness,'punct_density':n_punct/n_chars,'avg_sent_len':mean_len}
def pos_bigrams(tagged):
    tags = [t for _,t in tagged]; return list(zip(tags, tags[1:]))
def fit_pos_bigram_vocab(train_texts, top_k=36):
    counter = Counter()
    for text in tqdm(train_texts, desc='Fitting POS-bigram vocab (train only)'):
        counter.update(pos_bigrams(pos_tag(word_tokenize(text))))
    return [bg for bg,_ in counter.most_common(top_k)]
def grammatical_features(tagged, vocab):
    bigrams = pos_bigrams(tagged); total = len(bigrams); counts = Counter(bigrams)
    return {f'pos_{a}_{b}': (counts.get((a,b),0)/total if total>0 else 0.) for a,b in vocab}
def build_burrows_reference(train_human_texts, top_words):
    rates = {w: [] for w in top_words}
    for text in train_human_texts:
        tokens = [t.lower() for t in word_tokenize(text)]; n = max(len(tokens),1); freqs = Counter(tokens)
        for w in top_words: rates[w].append(freqs.get(w,0)/n)
    return ({w: float(np.mean(v)) for w,v in rates.items()}, {w: (float(np.std(v)) if np.std(v)>0 else 1.0) for w,v in rates.items()})
def burrows_delta(tokens, ref_mean, ref_std, top_words):
    n = max(len(tokens),1); freqs = Counter(tokens)
    diffs = [abs(((freqs.get(w,0)/n)-ref_mean.get(w,0.))/ref_std.get(w,1.)) for w in top_words]
    return float(np.mean(diffs)) if diffs else 0.
def function_word_ratios(tokens):
    n = max(len(tokens),1); freqs = Counter(t.lower() for t in tokens)
    return {f'func_{w}': freqs.get(w,0)/n for w in FUNCTION_WORDS}

class StylometricExtractor:
    def fit(self, train_texts, train_human_texts, n_bigrams=36, n_burrows_words=20):
        self.bigram_vocab = fit_pos_bigram_vocab(train_texts, n_bigrams)
        all_tok = [w.lower() for t in train_human_texts for w in word_tokenize(t)]
        self.burrows_words = [w for w,_ in Counter(all_tok).most_common(n_burrows_words) if w.isalpha()]
        self.ref_mean, self.ref_std = build_burrows_reference(train_human_texts, self.burrows_words)
        return self
    def transform(self, text, gpt2_ppl_fn=None):
        tokens, sentences = word_tokenize(text), sent_tokenize(text)
        tagged = pos_tag(tokens)
        feats = {}
        feats.update(lexical_features([t.lower() for t in tokens]))
        feats.update(syntactic_features(sentences))
        feats.update(grammatical_features(tagged, self.bigram_vocab))
        feats['burrows_delta'] = burrows_delta([t.lower() for t in tokens], self.ref_mean, self.ref_std, self.burrows_words)
        feats['gpt2_perplexity'] = gpt2_ppl_fn(text) if gpt2_ppl_fn else np.nan
        feats.update(function_word_ratios(tokens))
        return feats
    def transform_batch(self, texts, gpt2_ppl_fn=None, desc='Extracting'):
        rows = [self.transform(t, gpt2_ppl_fn) for t in tqdm(texts, desc=desc)]
        return pd.DataFrame(rows)

extractor_ckpt = 'shap_extractor'
if ckpt_exists(extractor_ckpt):
    extractor = ckpt_load(extractor_ckpt)
else:
    extractor = StylometricExtractor().fit(df_train['text'].tolist(), df_train[df_train.label==0]['text'].tolist())
    ckpt_save(extractor_ckpt, extractor)

feature_columns = (['hapax_ratio','yules_k','ttr','avg_word_len','sent_len_variance','burstiness',
                     'punct_density','avg_sent_len'] + [f'pos_{a}_{b}' for a,b in extractor.bigram_vocab] +
                    ['burrows_delta','gpt2_perplexity'] + [f'func_{w}' for w in FUNCTION_WORDS])
assert len(feature_columns) == 54
print('Feature dimension:', len(feature_columns))

_gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
_gpt2_lm = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()

@torch.no_grad()
def gpt2_perplexity(text, max_len=512):
    ids = _gpt2_tok(text, return_tensors='pt', truncation=True, max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1] < 2: return float('nan')
    loss = _gpt2_lm(ids, labels=ids).loss
    return float(torch.exp(loss).item())

X_style = {}
for name, d in [('train', df_train), ('testA', df_testA)]:
    ckpt_name = f'shap_style_{name}'
    if ckpt_exists(ckpt_name):
        X_style[name] = ckpt_load(ckpt_name)
    else:
        X_style[name] = extractor.transform_batch(d['text'].tolist(), gpt2_perplexity, f'style: {name}')
        ckpt_save(ckpt_name, X_style[name])

ppl_median = X_style['train']['gpt2_perplexity'].median()
for k in X_style: X_style[k]['gpt2_perplexity'] = X_style[k]['gpt2_perplexity'].fillna(ppl_median)

style_scaler = StandardScaler().fit(X_style['train'][feature_columns].values)
Xs = {k: style_scaler.transform(v[feature_columns].values) for k, v in X_style.items()}
print('Xs shapes:', {k: v.shape for k, v in Xs.items()})
print('Xs[train] range:', Xs['train'].min().round(2), 'to', Xs['train'].max().round(2))


## 5. SHAP on the self-consistent pipeline

In [ ]:
import shap
import matplotlib.pyplot as plt

shap_clf = LogisticRegression(max_iter=2000, random_state=42).fit(Xs['train'], y_train)
background = shap.sample(Xs['train'], min(200, Xs['train'].shape[0]), random_state=42)
explainer = shap.LinearExplainer(shap_clf, background)
shap_values = explainer.shap_values(Xs['testA'])

shap_summary = pd.DataFrame({
    'feature': feature_columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
    'mean_shap': shap_values.mean(axis=0),
    'lr_coef': shap_clf.coef_[0],
})
shap_summary['coef_shap_sign_match'] = np.sign(shap_summary.lr_coef) == np.sign(shap_summary.mean_shap)
shap_summary['direction'] = np.where(shap_summary.mean_shap > 0, '-> AI', '-> Human')
shap_summary = shap_summary.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features (self-consistent pipeline, real M4 data):')
print(shap_summary.head(15).to_string(index=False))
n_mismatch = (~shap_summary.coef_shap_sign_match).sum()
print(f'\n{n_mismatch}/{len(shap_summary)} features show a coef/SHAP sign mismatch '
      f'(expected under train->test covariate shift, verified mechanism -- not a bug).')

shap_summary.to_csv(f'{ARTIFACTS_DIR}/M4_shap_v2_selfconsistent.csv', index=False)
shap.summary_plot(shap_values, Xs['testA'], feature_names=feature_columns, show=False, max_display=15)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/M4_shap_v2_summary.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Reconciliation diagnostic — where does this pipeline diverge from `m4_stage2.pkl`?

Compares per-feature statistics against the original Stage-1/2 pickle's already-scaled
train features, to pinpoint (not necessarily explain) the 0.868-vs-0.791 discrepancy.

In [ ]:
import scipy.stats as stats

stage2_path = find_artifact(
    '/kaggle/input/datasets/amanhunyawr/m4-hybrid-stage2/m4_stage2.pkl',
    *glob.glob('/kaggle/input/*/m4_stage2.pkl'), *glob.glob('/kaggle/input/*/*/m4_stage2.pkl'))
print('Loading original stage-2 artifact from:', stage2_path)
with open(stage2_path, 'rb') as f:
    stage2_original = pickle.load(f)

Xs_original_train = stage2_original['Xs']['train']
Xs_v2_train = Xs['train']

if Xs_original_train.shape != Xs_v2_train.shape:
    print(f'NOTE: shape mismatch (original={Xs_original_train.shape}, this run={Xs_v2_train.shape}) -- '
          'likely different sample counts.')

# NOTE: comparing mean/std between two independently z-score-standardized arrays is
# tautological -- StandardScaler forces mean=0, std=1 on every column by construction,
# regardless of whether the underlying raw features actually agree. That comparison
# would show "no difference" even for completely unrelated data, so it can never
# actually catch a real discrepancy. Skew and kurtosis are NOT forced to any particular
# value by standardization (verified: skew is scale-invariant), so they can genuinely
# detect shape differences that survive both pipelines' scaling.
diag_rows = []
for i, col in enumerate(feature_columns):
    orig_col = Xs_original_train[:, i]
    v2_col = Xs_v2_train[:, i]
    diag_rows.append({
        'feature': col,
        'original_skew': stats.skew(orig_col), 'v2_skew': stats.skew(v2_col),
        'original_kurtosis': stats.kurtosis(orig_col), 'v2_kurtosis': stats.kurtosis(v2_col),
        'skew_abs_diff': abs(stats.skew(orig_col) - stats.skew(v2_col)),
        'kurtosis_abs_diff': abs(stats.kurtosis(orig_col) - stats.kurtosis(v2_col)),
    })
diag_df = pd.DataFrame(diag_rows).sort_values('skew_abs_diff', ascending=False)

print('Features where the two pipelines disagree most on distribution SHAPE (scale-invariant, top 10):')
print(diag_df.head(10)[['feature','original_skew','v2_skew','skew_abs_diff','original_kurtosis','v2_kurtosis']].round(3).to_string(index=False))
diag_df.round(4).to_csv(f'{ARTIFACTS_DIR}/M4_baseline_reconciliation_diagnostic_v2.csv', index=False)

print()
print('skew_abs_diff and kurtosis_abs_diff near 0 genuinely indicate agreement this time;')
print('large values (roughly >0.5 as a rule of thumb) indicate the two pipelines compute')
print('that feature meaningfully differently, even after standardization.')
